In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title Import thư viện cần thiết
import json
import os
import re
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# @title Cấu hình đường dẫn
INPUT_JSON       = '/content/drive/MyDrive/Data_mining/VietNamText/vietnam_only.json'
IMAGE_FOLDER     = '/content/drive/MyDrive/Data_mining/Wiki_VN_Images'                          # Folder chứa ảnh
SUMMARY_PARQUET  = '/content/drive/MyDrive/Data_mining/VietNamText/milvus_summaryVN.parquet'    # Đã có sẵn
CHUNKS_PARQUET   = '/content/drive/MyDrive/Data_mining/VietNamText/milvus_chunksVN.parquet'     # Đã có sẵn
OUTPUT_IMAGE_PQ  = '/content/drive/MyDrive/Data_mining/VietNamText/milvus_imagesVN.parquet'     # File output image vectors
OUTPUT_METADATA  = '/content/drive/MyDrive/Data_mining/VietNamText/img_retrieval_metadata.json'
DIM_IMG          = 2048                          # Dimension của ResNet-50 (avgpool output)
BATCH_SIZE       = 100

In [ ]:
# @title Load metadata từ vietnam_only.json
with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

# Tạo mapping id_dia_diem -> entry
id_to_entry = {str(item['id_dia_diem']): item for item in metadata}
print(f'Tổng số địa điểm: {len(id_to_entry)}')

# Tạo set các id_dia_diem hợp lệ
valid_ids = set(id_to_entry.keys())
print(f'Số id_dia_diem hợp lệ: {len(valid_ids)}')

Tổng số địa điểm: 574
Số id_dia_diem hợp lệ: 574


In [ ]:
# @title Khởi tạo ResNet-50 model (chỉ cần cho image)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Đang load ResNet-50 (pretrained ImageNet)...')
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
# Bỏ lớp FC cuối, chỉ lấy feature 2048-d từ avgpool
model_img = nn.Sequential(*list(resnet50.children())[:-1])  # output: (batch, 2048, 1, 1)
model_img = model_img.to(device).eval()

# Transform chuẩn cho ResNet-50
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print('ResNet-50 model loaded')

Device: cuda
Đang load ResNet-50 (pretrained ImageNet)...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 49.3MB/s]


ResNet-50 model loaded


In [ ]:
# tạo image embedding & parse filename

def get_image_embedding(image_path):
    """Tạo embedding cho 1 ảnh bằng ResNet-50, trả về vector 2048-d đã normalize."""
    try:
        image = Image.open(image_path).convert('RGB')
        img_tensor = preprocess(image).unsqueeze(0).to(device)
        with torch.no_grad():
            img_emb = model_img(img_tensor)           # (1, 2048, 1, 1)
            img_emb = img_emb.squeeze()                # (2048,)
            img_emb = img_emb / img_emb.norm()         # L2 normalize
        return img_emb.cpu().numpy()
    except Exception as e:
        print(f'Lỗi khi xử lý {image_path}: {e}')
        return None


def parse_image_filename(filename):
    """
    Parse filename theo format: iddiadiem_idảnh_tên_ảnh.ext
    Ví dụ: '164515_001_temple_view.jpg' -> id_dia_diem='164515', img_id='001', img_name='temple_view'
    """
    name_without_ext = os.path.splitext(filename)[0]
    parts = name_without_ext.split('_', 2)  # Split tối đa 2 lần

    if len(parts) >= 2:
        id_dia_diem = parts[0]
        img_id = parts[1]
        img_name = parts[2] if len(parts) > 2 else ''
        return id_dia_diem, img_id, img_name
    return None, None, None


def is_valid_image(filename):
    """Kiểm tra file có phải ảnh không."""
    exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff'}
    return os.path.splitext(filename)[1].lower() in exts

In [ ]:
import os

folder_path = "/content/drive/MyDrive/Data_mining/Wiki_Deep_Images"

file_count = 0
for root, dirs, files in os.walk(folder_path):
    file_count += len(files)

print("Tổng số file ảnh (kể cả folder con):", file_count)

In [ ]:
# 6. Scan folder A và xây dựng Image Vector Database

# Lấy danh sách tất cả ảnh trong folder A
all_image_files = [f for f in os.listdir(IMAGE_FOLDER) if is_valid_image(f)]
print(f'Tổng số ảnh trong folder "{IMAGE_FOLDER}": {len(all_image_files)}')

# Lọc chỉ giữ ảnh có id_dia_diem hợp lệ (tồn tại trong vietnam_only.json)
valid_image_files = []
skipped = 0
for f in all_image_files:
    id_dd, img_id, _ = parse_image_filename(f)
    if id_dd and id_dd in valid_ids:
        valid_image_files.append(f)
    else:
        skipped += 1

print(f'Ảnh hợp lệ (có id_dia_diem trong JSON): {len(valid_image_files)}')
print(f'Ảnh bị bỏ qua (id không khớp): {skipped}')

Tổng số ảnh trong folder "/content/drive/MyDrive/Data_mining/Wiki_VN_Images": 1600
Ảnh hợp lệ (có id_dia_diem trong JSON): 1600
Ảnh bị bỏ qua (id không khớp): 0


## Embedding imgs vectors

In [ ]:
# @title Tạo embedding cho tất cả ảnh hợp lệ và lưu vào parquet

image_records = []  # Lưu dữ liệu để export parquet
error_count = 0

pbar = tqdm(valid_image_files, desc='Encoding images')
for filename in pbar:
    id_dia_diem, img_id, img_name = parse_image_filename(filename)
    img_path = os.path.join(IMAGE_FOLDER, filename)

    emb = get_image_embedding(img_path)
    if emb is not None and emb.shape[0] == DIM_IMG:
        image_records.append({
            'image_id': f'{id_dia_diem}_{img_id}',   # ID duy nhất cho ảnh
            'id_dia_diem': int(id_dia_diem),             # Liên kết về địa điểm gốc
            'vector': emb.tolist(),                    # Vector ResNet-50 2048-d
            'filename': filename,                      # Tên file gốc
            'img_name': img_name                       # Phần tên mô tả
        })
    else:
        error_count += 1

    pbar.set_postfix({'OK': len(image_records), 'Err': error_count})

print(f'\nTạo embedding thành công: {len(image_records)} ảnh')
print(f' Lỗi: {error_count} ảnh')

In [ ]:
# @title Lưu Image Vector DB ra parquet (tương thích format Milvus)

df_images = pd.DataFrame(image_records)
df_images.to_parquet(OUTPUT_IMAGE_PQ, index=False)
print(f'✅ Đã lưu {len(df_images)} image vectors vào {OUTPUT_IMAGE_PQ}')
print(f'Columns: {list(df_images.columns)}')
df_images.head()

In [ ]:
# @title Load Image Vector DB để phục vụ retrieval

df_img = pd.read_parquet(OUTPUT_IMAGE_PQ)
image_vectors = np.array(df_img['vector'].tolist())  # shape: (N, 2048)
image_ids     = df_img['image_id'].tolist()
image_parents = df_img['id_dia_diem'].tolist()

# Mapping image_id -> parent_id (id_dia_diem)
img_id_to_parent = dict(zip(image_ids, image_parents))

print(f'Image vectors shape: {image_vectors.shape}')
print(f'Số ảnh trong DB: {len(image_ids)}')

In [ ]:
# @title Hàm retrieval bằng ảnh query

def retrieve_by_image(query_image_path, top_k=5):
    """
    Nhận đường dẫn ảnh query, trả về top-k ảnh gần nhất trong Image Vector DB.
    Returns: list of (image_id, similarity_score, id_dia_diem)
    """
    query_emb = get_image_embedding(query_image_path)
    if query_emb is None:
        print('Không thể tạo embedding cho ảnh query!')
        return []

    # Tính cosine similarity với tất cả ảnh trong DB
    sims = cosine_similarity([query_emb], image_vectors)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        results.append({
            'image_id': image_ids[i],
            'similarity': float(sims[i]),
            'id_dia_diem': image_parents[i],
            'title': id_to_entry.get(str(image_parents[i]), {}).get('title', 'N/A')
        })
    return results


def display_results(results):
    """Hiển thị kết quả retrieval."""
    print(f'\n{"="*60}')
    print(f'{"Rank":<6}{"Image ID":<20}{"Score":<10}{"ID Địa Điểm":<15}{"Title"}')
    print(f'{"="*60}')
    for rank, r in enumerate(results, 1):
        print(f'{rank:<6}{r["image_id"]:<20}{r["similarity"]:<10.4f}{r["id_dia_diem"]:<15}{r["title"]}')
    print(f'{"="*60}')

##Demo retrieval bằng ảnh



In [ ]:
# @title Demo retrieval bằng ảnh

# Thay đường dẫn ảnh query tại đây
query_path = os.path.join(IMAGE_FOLDER, valid_image_files[0])  # Dùng ảnh đầu tiên làm demo
print(f'Query image: {query_path}')

results = retrieve_by_image(query_path, top_k=5)
display_results(results)

## Load tất cả Vector DB (Summary + Chunk + Image)

In [ ]:
# Load tất cả vector DB từ parquet

# --- SUMMARY DB (đã có sẵn, dùng cho coarse search) ---
print('Loading Summary DB...')
df_sum = pd.read_parquet(SUMMARY_PARQUET)
summary_data = df_sum.to_dict('records')
summary_vecs = torch.tensor(np.array(df_sum['vector'].tolist())).float().to(device)
print(f'  Summary vectors: {summary_vecs.shape} (dtype: {summary_vecs.dtype})')

# --- CHUNK DB (đã có sẵn, dùng cho fine search) ---
print('Loading Chunk DB...')
df_chk = pd.read_parquet(CHUNKS_PARQUET)
chunk_data = df_chk.to_dict('records')
chunk_vecs = torch.tensor(np.array(df_chk['vector'].tolist())).float().to(device)
print(f'  Chunk vectors: {chunk_vecs.shape} (dtype: {chunk_vecs.dtype})')

# --- IMAGE DB (vừa tạo) ---
print('Loading Image DB...')
df_img = pd.read_parquet(OUTPUT_IMAGE_PQ)
image_vectors = np.array(df_img['vector'].tolist(), dtype=np.float32)
image_ids     = df_img['image_id'].tolist()
image_parents = df_img['id_dia_diem'].tolist()
img_id_to_parent = dict(zip(image_ids, image_parents))
print(f'  Image vectors: {image_vectors.shape}')

print('\n✅ All DBs loaded!')

Loading Summary DB...
  Summary vectors: torch.Size([574, 1024]) (dtype: torch.float32)
Loading Chunk DB...
  Chunk vectors: torch.Size([2364, 1024]) (dtype: torch.float32)
Loading Image DB...
  Image vectors: (1600, 2048)

✅ All DBs loaded!


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Data_mining/VietNamText/milvus_chunksVN.parquet"

df = pd.read_parquet(file_path)

print("Các cột (fields milvus_chunksVN.parquet):")
print(df.columns.tolist())

Các cột (fields milvus_chunksVN.parquet):
['chunk_id', 'parent_id', 'vector', 'content']


## Draft Text + Imgs Retrieval

In [ ]:
from sentence_transformers import SentenceTransformer, util
# --- Model E5 cho text retrieval (giống test.ipynb) ---
print('Đang load E5 model...')
model_text = SentenceTransformer('intfloat/multilingual-e5-large', device=device)


Đang load E5 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [ ]:
# @title Text Retrieval

def retrieve_by_text(query, top_k_summary=10, top_k_chunks=5):
    """
    Retrieval text theo flow coarse-to-fine (giống test.ipynb).
    Bước 1: E5 tìm top-k summary → lấy danh sách id_dia_diem
    Bước 2: E5 tìm top-k chunks trong các IDs đó
    Returns: list of dict {chunk_id, parent_id, id_dia_diem, similarity, content, title}
    """
    # --- Bước 1: Coarse search trên Summary ---
    with torch.no_grad():
        query_vec = model_text.encode(f'query: {query}', convert_to_tensor=True).to(device)

    s_scores = util.cos_sim(query_vec, summary_vecs)[0]
    top_s_indices = torch.topk(s_scores, k=min(top_k_summary, len(s_scores))).indices
    potential_ids = [str(summary_data[i]['id']) for i in top_s_indices]

    # --- Bước 2: Fine search trên Chunks thuộc top IDs ---
    valid_chunks = []
    valid_vecs = []
    for i, c in enumerate(chunk_data):
        if str(c['parent_id']) in potential_ids:
            valid_chunks.append(c)
            valid_vecs.append(chunk_vecs[i])

    if not valid_vecs:
        return []

    v_stack = torch.stack(valid_vecs).to(device)
    c_scores = util.cos_sim(query_vec, v_stack)[0]
    top_c = torch.topk(c_scores, k=min(top_k_chunks, len(c_scores)))

    results = []
    for score, idx in zip(top_c.values, top_c.indices):
        c = valid_chunks[idx]
        dd_id = int(c['parent_id'])
        results.append({
            'chunk_id': c.get('chunk_id', ''),
            'id_dia_diem': dd_id,
            'similarity': float(score),
            'content': c.get('content', ''),
            'title': id_to_entry.get(str(dd_id), {}).get('title', 'N/A')
        })
    return results

In [ ]:
# @title Image Retrieval

def retrieve_by_image(query_image_path, top_k=5):
    """Retrieval ảnh: query ảnh → top-k ảnh gần nhất trong Image DB."""
    query_emb = get_image_embedding(query_image_path)
    if query_emb is None:
        print('Không thể tạo embedding cho ảnh query!')
        return []
    sims = cosine_similarity([query_emb], image_vectors)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = []
    for i in top_idx:
        results.append({
            'image_id': image_ids[i],
            'similarity': float(sims[i]),
            'id_dia_diem': image_parents[i],
            'title': id_to_entry.get(str(image_parents[i]), {}).get('title', 'N/A')
        })
    return results

In [ ]:
# @title Hợp nhất kết quả text + image

def fuse_results(image_results, text_results, weight_image=0.4, weight_text=0.6):
    """Hợp nhất kết quả image + text theo id_dia_diem, weighted score."""
    scores = {}
    for r in image_results:
        dd_id = r['id_dia_diem']
        scores[dd_id] = scores.get(dd_id, 0) + weight_image * r['similarity']
    for r in text_results:
        dd_id = r['id_dia_diem']
        scores[dd_id] = scores.get(dd_id, 0) + weight_text * r['similarity']
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
# @title Demo: Text Retrieval

query = 'Where is Yên Hòa located?'
print(f'🔍 Query: "{query}"')
print('='*70)

text_results = retrieve_by_text(query, top_k_summary=10, top_k_chunks=5)
print(f'\n📝 Text Retrieval Results:')
for rank, r in enumerate(text_results, 1):
    print(f'  [{rank}] Score: {r["similarity"]:.4f} | ID: {r["id_dia_diem"]} | {r["title"]}')
    print(f'      Content: {r["content"][:120]}...')
    print()

🔍 Query: "Where is Yên Hòa located?"

📝 Text Retrieval Results:
  [1] Score: 0.8840 | ID: 164891 | Yên Hòa
      Content: [Yên Hòa - General Information] Yên Hòa [iən˧˧:hwa̤ː˨˩] is a ward of Hanoi the capital city in the Red River Delta of Vi...

  [2] Score: 0.8644 | ID: 164273 | Ninh Hòa
      Content: [Ninh Hòa - General Information] Ninh Hòa is a district-level town (thị xã) of Khánh Hòa province in the South Central C...

  [3] Score: 0.8638 | ID: 164891 | Yên Hòa
      Content: [Yên Hòa - History] Its name Yên Hòa (Hán: 安和, Nôm: 𭴣和) is originally a combination of seven localities that have many c...

  [4] Score: 0.8431 | ID: 164891 | Yên Hòa
      Content: [Yên Hòa - Topography] Yên Hòa Ward situates roughly to the Southwest area of urban Hanoi. According to the content of t...

  [5] Score: 0.8410 | ID: 164891 | Yên Hòa
      Content: [Yên Hòa - Culture] Yên Hòa Ward has long been dubbed one of the three core areas of tradition in Hà Nội the capital. Th...



In [ ]:
# 14. Demo: Image Retrieval

query_path = os.path.join(IMAGE_FOLDER, valid_image_files[0])
print(f'Query image: {query_path}')
print('='*70)

img_results = retrieve_by_image(query_path, top_k=5)
print(f'\n Image Retrieval Results:')
for rank, r in enumerate(img_results, 1):
    print(f'  [{rank}] Score: {r["similarity"]:.4f} | ID: {r["id_dia_diem"]} | {r["title"]} | img: {r["image_id"]}')

Query image: /content/drive/MyDrive/Data_mining/Wiki_VN_Images/164891_71_Yên_Hòa_Viet_Nam.jpg

 Image Retrieval Results:
  [1] Score: 1.0000 | ID: 164891 | Yên Hòa | img: 164891_71
  [2] Score: 0.6597 | ID: 164735 | Liên Chiểu district | img: 164735_1
  [3] Score: 0.6393 | ID: 164170 | Ho Chi Minh City | img: 164170_39
  [4] Score: 0.6218 | ID: 164891 | Yên Hòa | img: 164891_40
  [5] Score: 0.6176 | ID: 164305 | Mỹ Tho | img: 164305_18


In [ ]:
# 15. Demo: Fusion (Text + Image)

fused = fuse_results(img_results, text_results, weight_image=0.6, weight_text=0.4)
print('\n Fused Results (Text + Image):')
print('='*50)
for rank, (dd_id, score) in enumerate(fused[:10], 1):
    title = id_to_entry.get(str(dd_id), {}).get('title', 'N/A')
    print(f'  [{rank}] Score: {score:.4f} | ID: {dd_id} | {title}')


 Fused Results (Text + Image):
  [1] Score: 2.3458 | ID: 164891 | Yên Hòa
  [2] Score: 0.3958 | ID: 164735 | Liên Chiểu district
  [3] Score: 0.3836 | ID: 164170 | Ho Chi Minh City
  [4] Score: 0.3705 | ID: 164305 | Mỹ Tho
  [5] Score: 0.3457 | ID: 164273 | Ninh Hòa


## Save metadata img_retrieval

In [ ]:
#Lưu metadata phục vụ downstream (RAG / chatbot)

def save_retrieval_metadata(path=None):
    """
    Lưu metadata cho các entry có ảnh, bao gồm:
    wiki_link, id_wiki, summary_wiki, images, img_ids
    """
    if path is None:
        path = OUTPUT_METADATA

    # Lấy danh sách id_dia_diem có ảnh trong DB
    ids_with_images = set(image_parents)

    out = []
    for dd_id in ids_with_images:
        entry = id_to_entry.get(str(dd_id), {})
        # Lấy danh sách image_id thuộc về địa điểm này
        entry_img_ids = [img_id for img_id, parent in img_id_to_parent.items() if parent == dd_id]

        out.append({
            'wiki_link': entry.get('url', ''),
            'id_wiki': dd_id,
            'title': entry.get('title', ''),
            'summary_wiki': entry.get('summary_text', ''),
            'img_ids': entry_img_ids,
            'num_images': len(entry_img_ids)
        })

    with open(path, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f'✅ Lưu metadata {len(out)} địa điểm vào {path}')

save_retrieval_metadata()